In [1]:
!pip install transformers datasets

In [2]:
# ------------------------------------
#  Mount Google Drive
# ------------------------------------

# Import library to access Google Drive in Colab
from google.colab import drive

# Mount Google Drive to access files stored in your drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ------------------------------------
#  Import Required Libraries
# ------------------------------------


# Import pandas library for handling and manipulating datasets (CSV files, tables, etc.)
import pandas as pd

# Import PyTorch library for building and training deep learning models
import torch

# Import DataLoader to efficiently load and batch the dataset during training
from torch.utils.data import DataLoader

# Import DistilBERT tokenizer to convert text into numerical tokens that the model can understand
from transformers import DistilBertTokenizerFast

# Import DistilBERT model for sequence classification (used here for fake news detection)
from transformers import DistilBertForSequenceClassification

# Import AdamW optimizer used for updating model weights during training
from torch.optim import AdamW

In [5]:
# ------------------------------------
#  Load Dataset and Split Features
# ------------------------------------


# Load training and testing datasets
train_df = pd.read_csv('/content/drive/MyDrive/Fake News Detection /Processed Data/train_dataset.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Fake News Detection /Processed Data/test_dataset.csv')

# Input text for training
X_train = train_df['content']

# Labels for training
Y_train = train_df['label']

# Input text for testing
X_test = test_df['content']

# Labels for testing
Y_test = test_df['label']

In [7]:
# Load Member-2 Tokenizer for Model
# Load tokenizer from the saved 'tokenizer' folder
tokenizer = DistilBertTokenizerFast.from_pretrained('/content/drive/MyDrive/Fake News Detection /Processed Data/Tokenizer')

In [ ]:

# ------------------------------------
#  Tokenize Training and Test Text
# ------------------------------------
# Ensure X_train is also cleaned of potential NaN values and converted to a list of strings
X_train_list = X_train.fillna("").astype(str).values.tolist()
X_test_list = X_test.fillna("").astype(str).values.tolist() # Updated line to convert X_test to list of strings and handle NaNs

train_encodings = tokenizer(
X_train_list, # convert training text to list
truncation=True, # truncate long text
padding = "max_length", # pad shorter text
max_length=256 # maximum token length
)

test_encodings = tokenizer(
X_test_list, # convert test text to list
truncation=True, # truncate long text
padding="max_length", # pad shorter text
max_length=256 # maximum token length
)


In [ ]:
import torch
# ------------------------------------
#  Create Data Loaders
# ------------------------------------
# Define a custom Dataset class for PyTorch
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Fix the UserWarning by using .detach().clone() for labels
        item['labels'] = self.labels[idx].detach().clone().to(torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# Convert labels to PyTorch tensors
y_train = torch.tensor(Y_train.values, dtype=torch.long)
y_test = torch.tensor(Y_test.values, dtype=torch.long)

# DataLoader is a PyTorch utility that helps load datasets efficiently.
# It automatically divides the dataset into smaller batches and feeds them to the model.
# This improves training speed and memory usage.

train_dataset = NewsDataset(train_encodings, y_train)
test_dataset  = NewsDataset(test_encodings, y_test)
# Create a DataLoader for the training dataset
# - batch_size=16 → The model will process 16 samples at a time
# - shuffle=True → Randomizes the order of data every epoch
#   This improves training by preventing the model from learning patterns
#   based on the sequence of the dataset
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# Create a DataLoader for the test dataset
# - batch_size=16 → Test data is also evaluated in batches of 16
# - shuffle is not used because the order of test data should remain fixed
#   for consistent evaluation and reproducibility
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# --------------------------------------
#  Load DistilBERT Classification Model
# --------------------------------------

# Load a pretrained DistilBERT model that is already trained on large text datasets
# and adapt it for a sequence classification task

# "distilbert-base-uncased"
# - base → standard model size
# - uncased → converts all text to lowercase (case-insensitive)

# num_labels=2
# - Specifies the number of output classes
# - Here: Fake News (0) and Real News (1)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# ------------------------------------
#  Set Device for Training
# ------------------------------------
# torch.device() is used to specify where the model will run:
# either on the GPU or the CPU.
# "cuda" refers to the GPU (Graphics Processing Unit).
# GPUs can perform many calculations in parallel, making
# deep learning training much faster than using a CPU.
# torch.cuda.is_available()
# checks if a CUDA-compatible GPU is available on the system.
# If a GPU is available → use GPU ("cuda")
# Otherwise → fall back to CPU ("cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Move the model to the selected device
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# ------------------------------------
#  Define Optimizer
# ------------------------------------

# AdamW optimizer is commonly used for training transformer models like DistilBERT
# It updates the model's weights during backpropagation to minimize the loss function

# model.parameters()
# - Provides all trainable parameters (weights and biases) of the model
# - The optimizer will update these parameters during training

# lr = 2e-5 (learning rate = 0.00002)
# - The learning rate controls how much the model weights change during each update
# - A very small learning rate is used when fine-tuning pretrained transformer models
# - Large learning rates can destroy the pretrained knowledge learned from large datasets
# - Research and Hugging Face documentation recommend values between 2e-5 and 5e-5
#   for stable and effective fine-tuning of BERT-based models

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
# ------------------------------------
#  Training Loop
# ------------------------------------

epochs = 3  # number of training epochs

for epoch in range(epochs):  # loop through each training epoch

    model.train()  # set model to training mode

    for i,batch in enumerate(train_loader):
        if i % 100 == 0:
          print(f"Epoch {epoch+1},Batch{i}")  # iterate through batches of training data

          optimizer.zero_grad()  # reset gradients

          input_ids = batch["input_ids"].to(device)  # move input ids to device
          attention_mask = batch["attention_mask"].to(device)  # move attention mask to device
          labels = batch["labels"].to(device)  # move labels to device

          outputs = model(
              input_ids=input_ids,
              attention_mask=attention_mask,
              labels=labels
          )  # forward pass

          loss = outputs.loss  # calculate loss
          loss.backward()  # backpropagation

          optimizer.step()  # update model weights

    print(f"Epoch {epoch+1} completed")  # print epoch completion


Epoch 1,Batch0
Epoch 1,Batch100
Epoch 1,Batch200
Epoch 1,Batch300
Epoch 1,Batch400
Epoch 1,Batch500
Epoch 1,Batch600
Epoch 1,Batch700
Epoch 1,Batch800
Epoch 1,Batch900
Epoch 1,Batch1000
Epoch 1,Batch1100
Epoch 1 completed
Epoch 2,Batch0
Epoch 2,Batch100
Epoch 2,Batch200
Epoch 2,Batch300
Epoch 2,Batch400
Epoch 2,Batch500
Epoch 2,Batch600
Epoch 2,Batch700
Epoch 2,Batch800
Epoch 2,Batch900
Epoch 2,Batch1000
Epoch 2,Batch1100
Epoch 2 completed
Epoch 3,Batch0
Epoch 3,Batch100
Epoch 3,Batch200
Epoch 3,Batch300
Epoch 3,Batch400
Epoch 3,Batch500
Epoch 3,Batch600
Epoch 3,Batch700
Epoch 3,Batch800
Epoch 3,Batch900
Epoch 3,Batch1000
Epoch 3,Batch1100
Epoch 3 completed


In [ ]:
# ------------------------------------
#  Save Trained Model and Tokenizer
# ------------------------------------

# Save the trained DistilBERT model to the 'fake_news_model' folder
model.save_pretrained("/content/drive/MyDrive/Fake News Detection /Data/Model Training")

# Save the tokenizer so it can be used later for prediction
tokenizer.save_pretrained("/content/drive/MyDrive/Fake News Detection /Data/Model Training")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Fake News Detection /Data/Model Training/tokenizer_config.json',
 '/content/drive/MyDrive/Fake News Detection /Data/Model Training/tokenizer.json')